# Search-2-Uninformed (C#) : Algorithmes de Recherche Non Informée

**Navigation** : [<< Espaces d'états (Search-1)](Search-1-StateSpace.ipynb) | [Index Partie 1](README.md) | [Recherche informée (Search-3) >>](Search-3-Informed.ipynb)

**Jumeau .NET** du notebook Python [Search-2-Uninformed.ipynb](Search-2-Uninformed.ipynb) — port fidèle en C# .NET 9.0, dans le cadre du marathon de parité .NET ⇄ Python (#4956). Les quatre algorithmes fondamentaux de la recherche **non informée** sont réimplémentés à l'aide de la BCL .NET (`Queue`, `Stack`, `PriorityQueue`), sans bibliothèque externe.

---

## 1. Introduction (~5 min)

Un algorithme de recherche **non informé** (uninformed / blind search) explore l'espace d'états **sans knowledge supplémentaire** sur le problème — aucune heuristique, aucune estimation de la distance au but. Sa stratégie est entièrement déterminée par l'ordre d'expansion des nœuds.

| Algorithme | Structure | Complétude | Optimalité | Complexité mémoire |
|------------|-----------|------------|------------|--------------------|
| **BFS** (largeur) | File FIFO | Oui (graphe fini) | Oui si coûts unitaires | O(b^d) |
| **DFS** (profondeur) | Pile LIFO | Non (cycles) | Non | O(b·m) |
| **UCS** (coût uniforme) | File de priorité | Oui | Oui (coûts ≥ 0) | O(b^(1+C*/ε)) |
| **IDDFS** (profondeur itérée) | DFS borné itératif | Oui | Oui si coûts unitaires | O(b·d) |

Nous illustrerons chaque algorithme sur le même problème de référence : trouver un itinéraire entre villes françaises sur un graphe routier pondéré par les distances kilométriques.

## 2. Cadre générique de recherche

Avant d'implémenter chaque algorithme, définissons les briques communes : un `Node` (nœud de recherche), une classe abstraite `Problem`, et un `GraphProblem` concret pour les graphes pondérés.

In [1]:
// --- Définition du nœud de recherche et du problème générique ---
using System;
using System.Collections.Generic;
using System.Linq;

// Noeud dans l'arbre de recherche : état + parent + coût g depuis la racine.
public record Node(string State, Node Parent, int Action, double G)
{
    // Reconstruction du chemin depuis la racine jusqu'à ce noeud.
    public List<Node> Path()
    {
        var path = new List<Node>();
        for (Node n = this; n != null; n = n.Parent) path.Add(n);
        path.Reverse();
        return path;
    }
    public int Depth => Parent is null ? 0 : Parent.Depth + 1;
}

// Problème de recherche abstrait (interface unifiée pour tous les algorithmes).
public abstract class Problem
{
    public string Initial { get; }
    public string Goal { get; }
    protected Problem(string initial, string goal) { Initial = initial; Goal = goal; }
    public abstract IEnumerable<(string next, double cost)> Actions(string state);
    public bool IsGoal(string state) => state == Goal;
}

Console.WriteLine("Cadre Node + Problem défini.");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Cadre Node + Problem défini.


### Problème concret : villes françaises

Définissons le graphe routier des villes françaises (12 villes, distances kilométriques symétriques). C'est notre problème de référence pour toute la suite.

In [2]:
// --- Problème de cheminement sur graphe pondéré + carte des villes françaises ---
public class GraphProblem : Problem
{
    private readonly Dictionary<string, Dictionary<string, double>> _graph;
    public GraphProblem(string initial, string goal,
                        Dictionary<string, Dictionary<string, double>> graph)
        : base(initial, goal) => _graph = graph;

    public override IEnumerable<(string next, double cost)> Actions(string state)
    {
        if (_graph.TryGetValue(state, out var nbrs))
            foreach (var kv in nbrs)
                yield return (kv.Key, kv.Value);
    }
}

// Carte routière française (distances en km, arêtes symétriques).
var franceGraph = new Dictionary<string, Dictionary<string, double>>
{
    ["Paris"]      = new(){ ["Lyon"]=465, ["Lille"]=225, ["Strasbourg"]=490, ["Nantes"]=385, ["Bordeaux"]=585 },
    ["Lyon"]       = new(){ ["Paris"]=465, ["Marseille"]=315, ["Grenoble"]=110, ["Strasbourg"]=490 },
    ["Marseille"]  = new(){ ["Lyon"]=315, ["Toulouse"]=405, ["Nice"]=200, ["Montpellier"]=170 },
    ["Toulouse"]   = new(){ ["Marseille"]=405, ["Bordeaux"]=245, ["Montpellier"]=245 },
    ["Bordeaux"]   = new(){ ["Paris"]=585, ["Toulouse"]=245, ["Nantes"]=340 },
    ["Nantes"]     = new(){ ["Paris"]=385, ["Bordeaux"]=340, ["Rennes"]=110 },
    ["Rennes"]     = new(){ ["Nantes"]=110, ["Lille"]=600 },
    ["Lille"]      = new(){ ["Paris"]=225, ["Rennes"]=600, ["Strasbourg"]=530 },
    ["Strasbourg"] = new(){ ["Paris"]=490, ["Lyon"]=490, ["Lille"]=530 },
    ["Nice"]       = new(){ ["Marseille"]=200, ["Grenoble"]=330 },
    ["Grenoble"]   = new(){ ["Lyon"]=110, ["Nice"]=330 },
    ["Montpellier"]= new(){ ["Marseille"]=170, ["Toulouse"]=245 },
};

int roads = franceGraph.Values.Sum(d => d.Count) / 2;
Console.WriteLine($"Graphe des villes françaises chargé : {franceGraph.Count} villes, {roads} routes.");

Graphe des villes françaises chargé : 12 villes, 18 routes.


### Utilitaire d'affichage

Encapsulons le résultat d'une recherche (chemin, coût, nombre de nœuds explorés) dans un petit enregistrement, avec un formateur textuel.

In [3]:
// --- Résultat de recherche + formateur ---
public record SearchResult(Node Goal, int Explored, string Name)
{
    public double Cost => Goal?.G ?? double.PositiveInfinity;
    public int Length => Goal?.Depth ?? -1;
    public bool Found => Goal is not null;
    public string PathString => Found ? string.Join(" -> ", Goal.Path().Select(n => n.State)) : "—";

    public void Print()
    {
        if (!Found) { Console.WriteLine($"{Name} : AUCUN CHEMIN TROUVÉ ({Explored} noeuds explorés)"); return; }
        Console.WriteLine($"{Name} : {PathString}");
        Console.WriteLine($"         coût = {Cost,7:F0} km | {Length} sauts | {Explored} noeuds explorés");
    }
}

Console.WriteLine("SearchResult défini.");

SearchResult défini.


## 3. Recherche en Largeur (BFS — Breadth-First Search)

BFS explore l'arbre **niveau par niveau** : tous les nœuds de profondeur *k* sont développés avant ceux de profondeur *k+1*. Implémentée avec une **file FIFO** (`Queue<T>`), elle garantit de trouver le chemin en **le moins de sauts** (optimal seulement si tous les coûts sont unitaires).

In [4]:
// --- BFS : recherche en largeur (file FIFO) + ensemble des visités ---
public static SearchResult BFS(Problem p, string start = null)
{
    string s0 = start ?? p.Initial;
    var frontier = new Queue<Node>();
    frontier.Enqueue(new Node(s0, null, 0, 0));
    var explored = new HashSet<string> { s0 };
    int exploredCount = 0;
    while (frontier.Count > 0)
    {
        var node = frontier.Dequeue();
        exploredCount++;
        if (p.IsGoal(node.State)) return new SearchResult(node, exploredCount, "BFS");
        foreach (var (next, cost) in p.Actions(node.State))
            if (!explored.Contains(next) && !frontier.Any(n => n.State == next))
            {
                explored.Add(next);
                frontier.Enqueue(new Node(next, node, 0, node.G + cost));
            }
    }
    return new SearchResult(null, exploredCount, "BFS");
}

var pBS = new GraphProblem("Bordeaux", "Strasbourg", franceGraph);
var bfsRes = BFS(pBS);
bfsRes.Print();

BFS : Bordeaux -> Paris -> Strasbourg


         coût =    1075 km | 2 sauts | 7 noeuds explorés


### Interprétation — BFS sur les villes françaises

BFS trouve le chemin en **le moins de sauts** (Bordeaux → Paris → Strasbourg, 2 sauts). Mais ce n'est pas forcément le **moins coûteux** en kilomètres : BFS ignore les poids des arêtes. C'est précisément ce que corrige UCS.

## 4. Recherche en Profondeur (DFS — Depth-First Search)

DFS plonge **le plus profondément possible** avant de remonter. Implémentée avec une **pile LIFO** (`Stack<T>`), elle est économe en mémoire mais ne garantit ni l'optimalité ni, sur un graphe cyclique sans suivi des visités, la terminaison.

In [5]:
// --- DFS : recherche en profondeur (pile LIFO) + visités ---
public static SearchResult DFS(Problem p, string start = null)
{
    string s0 = start ?? p.Initial;
    var frontier = new Stack<Node>();
    frontier.Push(new Node(s0, null, 0, 0));
    var explored = new HashSet<string>();
    int exploredCount = 0;
    while (frontier.Count > 0)
    {
        var node = frontier.Pop();
        if (explored.Contains(node.State)) continue;
        explored.Add(node.State);
        exploredCount++;
        if (p.IsGoal(node.State)) return new SearchResult(node, exploredCount, "DFS");
        // empiler dans l'ordre inverse pour conserver l'ordre d'expansion naturel
        var actions = p.Actions(node.State).Reverse().ToList();
        foreach (var (next, cost) in actions)
            if (!explored.Contains(next))
                frontier.Push(new Node(next, node, 0, node.G + cost));
    }
    return new SearchResult(null, exploredCount, "DFS");
}

var dfsRes = DFS(pBS);
dfsRes.Print();

DFS : Bordeaux -> Paris -> Lyon -> Strasbourg


         coût =    1540 km | 3 sauts | 9 noeuds explorés


### Interprétation — DFS vs BFS

DFS suit un chemin en profondeur avant tout autre : il descend jusqu'au but dès qu'il le peut, sans garantie de minimiser ni les sauts ni le coût. Le chemin retourné dépend de l'ordre d'expansion des voisins — c'est pourquoi DFS n'est pas optimal.

### Exercice 1 : DFS avec limite de profondeur

**Objectif** : implémentez `DfsLimited(problem, maxDepth)` — une variante de DFS qui n'explore pas au-delà de `maxDepth`. C'est la brique de base d'IDDFS (section 6). *Indice* : ajoutez un test sur `node.Depth` avant d'empiler les successeurs.

In [6]:
// --- Exercice 1 : DFS avec limite de profondeur (STUB) ---
// TODO étudiant : implémentez DfsLimited(p, maxDepth) qui retourne un SearchResult.
// Il doit se comporter comme DFS mais refuser d'empiler un successeur dont la profondeur
// dépasserait maxDepth. Le stub ci-dessous retourne null — à compléter.
public static SearchResult DfsLimited(Problem p, int maxDepth)
{
    // Indice : structure identique à DFS, avec la garde :
    //   if (node.Depth + 1 > maxDepth) continue;
    return null;  // TODO étudiant
}

// Test (à décommenter une fois complété) :
// Console.WriteLine("DFS limité (profondeur 3) Bordeaux -> Strasbourg :");
// DfsLimited(pBS, 3)?.Print();

### Illustration : ordre d'expansion BFS vs DFS sur un arbre

Pour visualiser la différence, construisons un petit arbre binaire et observons l'ordre de visite de chaque algorithme.

In [7]:
// --- Ordre d'expansion BFS vs DFS sur un arbre binaire de profondeur 3 ---
// Noeuds étiquetets A (racine), B-C (niveau 1), D-E-F-G (niveau 2), H..O (niveau 3).
var tree = new Dictionary<string, List<string>>
{
    ["A"] = new(){ "B", "C" },
    ["B"] = new(){ "D", "E" }, ["C"] = new(){ "F", "G" },
    ["D"] = new(){ "H", "I" }, ["E"] = new(){ "J", "K" },
    ["F"] = new(){ "L", "M" }, ["G"] = new(){ "N", "O" },
};
// Snapshot des clés avant mutation : on ajoute les feuilles pendant l'itération.
foreach (var k in tree.Keys.ToList())
    foreach (var c in tree[k])
        if (!tree.ContainsKey(c)) tree[c] = new List<string>();

List<string> BfsOrder(Dictionary<string,List<string>> g, string root)
{
    var order = new List<string>();
    var q = new Queue<string>(); q.Enqueue(root);
    var seen = new HashSet<string>{ root };
    while (q.Count > 0)
    {
        var n = q.Dequeue(); order.Add(n);
        foreach (var c in g.GetValueOrDefault(n, new())) if (seen.Add(c)) q.Enqueue(c);
    }
    return order;
}
List<string> DfsOrder(Dictionary<string,List<string>> g, string root)
{
    var order = new List<string>();
    var st = new Stack<string>(); st.Push(root);
    var seen = new HashSet<string>();
    while (st.Count > 0)
    {
        var n = st.Pop();
        if (!seen.Add(n)) continue;
        order.Add(n);
        foreach (var c in Enumerable.Reverse(g.GetValueOrDefault(n, new()))) st.Push(c);
    }
    return order;
}

Console.WriteLine("BFS : " + string.Join(", ", BfsOrder(tree, "A")));
Console.WriteLine("DFS : " + string.Join(", ", DfsOrder(tree, "A")));

BFS : A, B, C, D, E, F, G, H, I, J, K, L, M, N, O


DFS : A, B, D, H, I, E, J, K, C, F, L, M, G, N, O


### Interprétation — Ordres d'exploration

- **BFS** visite `A, B, C, D, E, F, G, H, …` : par niveaux, largeur d'abord.
- **DFS** visite `A, B, D, H, I, E, J, K, …` : descend branche gauche jusqu'au fond, puis remonte.

Ces deux ordres sont la signature des deux stratégies : BFS garantit le plus court chemin en nombre d'arêtes ; DFS garantit une faible empreinte mémoire.

## 5. Recherche à Coût Uniforme (UCS — Uniform Cost Search)

UCS généralise BFS aux **graphes pondérés** : au lieu d'expanser le nœud le moins profond, on expansse le nœud de **coût g minimal** (coût cumulé depuis la source). Implémentée avec une **file de priorité** (`PriorityQueue<TElement, TPriority>`), UCS garantit l'optimalité dès que les coûts sont ≥ 0.

In [8]:
// --- UCS : recherche à coût uniforme (file de priorité sur g) ---
public static SearchResult UCS(Problem p, string start = null)
{
    string s0 = start ?? p.Initial;
    var frontier = new PriorityQueue<Node, double>();
    frontier.Enqueue(new Node(s0, null, 0, 0), 0);
    var bestG = new Dictionary<string, double> { [s0] = 0 };
    int exploredCount = 0;
    while (frontier.Count > 0)
    {
        var node = frontier.Dequeue();
        exploredCount++;
        if (p.IsGoal(node.State)) return new SearchResult(node, exploredCount, "UCS");
        foreach (var (next, cost) in p.Actions(node.State))
        {
            double g = node.G + cost;
            if (!bestG.TryGetValue(next, out double prev) || g < prev)
            {
                bestG[next] = g;
                frontier.Enqueue(new Node(next, node, 0, g), g);
            }
        }
    }
    return new SearchResult(null, exploredCount, "UCS");
}

var ucsRes = UCS(pBS);
ucsRes.Print();

UCS : Bordeaux -> Paris -> Strasbourg


         coût =    1075 km | 2 sauts | 13 noeuds explorés


### Interprétation — UCS trouve le chemin optimal en coût

Sur Bordeaux → Strasbourg, UCS et BFS convergent ici vers le même chemin (le graphe est quasi-métrique sur cette paire : le chemin en le moins de sauts est aussi le moins coûteux). Ce n'est **pas** toujours le cas — c'est l'objet de la démonstration suivante.

### Démonstration Prong B : quand UCS bat BFS strictement

Prenons le trajet **Rennes → Strasbourg**. BFS privilégie le chemin en le moins de sauts (Rennes → Lille → Strasbourg, 2 sauts), qui coûte 600 + 530 = **1130 km**. UCS, lui, examine le coût et trouve un chemin à 3 sauts strictement moins cher : Rennes → Nantes → Paris → Strasbourg = 110 + 385 + 490 = **985 km** (−13 %). C'est l'apport réel d'UCS.

In [9]:
// --- Démonstration Prong B : UCS strictement meilleur que BFS sur Rennes -> Strasbourg ---
var pRS = new GraphProblem("Rennes", "Strasbourg", franceGraph);
Console.WriteLine("Trajet Rennes -> Strasbourg :");
BFS(pRS, "Rennes").Print();
UCS(pRS, "Rennes").Print();
Console.WriteLine(">>> UCS économise 1130 - 985 = 145 km (-13 %) en empruntant un chemin À 3 sauts moins cher que le chemin à 2 sauts de BFS.");

Trajet Rennes -> Strasbourg :


BFS : Rennes -> Lille -> Strasbourg


         coût =    1130 km | 2 sauts | 6 noeuds explorés


UCS : Rennes -> Nantes -> Paris -> Strasbourg


         coût =     985 km | 3 sauts | 9 noeuds explorés


>>> UCS économise 1130 - 985 = 145 km (-13 %) en empruntant un chemin À 3 sauts moins cher que le chemin à 2 sauts de BFS.


### Exercice 2 : Analyser l'écart de coût entre BFS et UCS

**Objectif** : identifiez un autre trajet dans le graphe où BFS ne trouve pas le chemin optimal en coût, et quantifiez l'écart. *Indice* : cherchez une paire de villes directement reliées par une arête coûteuse, pour laquelle un détour par 2 arêtes bon marché est globalement moins cher.

In [10]:
// --- Exercice 2 : Analyser l'écart BFS vs UCS (STUB) ---
// TODO étudiant : choisissez une paire (origine, destination) dans franceGraph,
// lancez BFS et UCS, puis affichez l'écart de coût en km.
// Exemple de structure attendue :
//   var p = new GraphProblem("<villeA>", "<villeB>", franceGraph);
//   var b = BFS(p); var u = UCS(p);
//   Console.WriteLine($"Écart : {b.Cost - u.Cost:F0} km");
string villeA = "Rennes";   // TODO étudiant : à remplacer
string villeB = "Nice";     // TODO étudiant : à remplacer
Console.WriteLine("Exercice 2 à compléter — choix initial (non optimal) : " + villeA + " -> " + villeB);

Exercice 2 à compléter — choix initial (non optimal) : Rennes -> Nice


## 6. Recherche en Profondeur Itérée (IDDFS — Iterative Deepening DFS)

IDDFS combine les garanties de BFS (complétude, optimalité en coûts unitaires) avec la **frugalité mémoire de DFS**. On lance une série de recherches DFS bornées à profondeur 0, 1, 2, … jusqu'à trouver le but. Le coût de la re-exploration reste modeste (overhead logarithmique).

In [11]:
// --- IDDFS : profondeur limitée + approfondissement itératif ---
// DLS : DFS borné à limit (retourne le noeud-but ou null).
public static Node DepthLimitedSearch(Problem p, int limit, out int explored)
{
    explored = 0;
    var frontier = new Stack<Node>();
    frontier.Push(new Node(p.Initial, null, 0, 0));
    while (frontier.Count > 0)
    {
        var node = frontier.Pop();
        explored++;
        if (p.IsGoal(node.State)) return node;
        if (node.Depth < limit)
            foreach (var (next, cost) in p.Actions(node.State).Reverse())
                frontier.Push(new Node(next, node, 0, node.G + cost));
    }
    return null;
}

public static SearchResult IDDFS(Problem p, int maxLimit = 20)
{
    int totalExplored = 0;
    for (int limit = 0; limit <= maxLimit; limit++)
    {
        var goal = DepthLimitedSearch(p, limit, out int explored);
        totalExplored += explored;
        if (goal is not null) return new SearchResult(goal, totalExplored, $"IDDFS");
    }
    return new SearchResult(null, totalExplored, "IDDFS");
}

var iddfsRes = IDDFS(pBS);
iddfsRes.Print();

IDDFS : Bordeaux -> Paris -> Strasbourg


         coût =    1075 km | 2 sauts | 10 noeuds explorés


### Interprétation — IDDFS et l'overhead de re-exploration

IDDFS re-explore les nœuds peu profonds à chaque itération. Sur un arbre de facteur de branchement *b* et profondeur solution *d*, l'overhead théorique est `b/(b−1)` (asymptotiquement négligeable). C'est le prix à payer pour une empreinte mémoire en `O(b·d)` au lieu de `O(b^d)` pour BFS.

## 7. Comparaison des quatre algorithmes

Comparons BFS, DFS, UCS et IDDFS sur plusieurs trajets. Le tableau ci-dessous résume coût (km), nombre de sauts et nœuds explorés.

In [12]:
// --- Comparaison complète sur plusieurs trajets ---
var testCases = new (string from, string to)[]
{
    ("Bordeaux", "Strasbourg"),
    ("Rennes", "Strasbourg"),
    ("Rennes", "Nice"),
    ("Bordeaux", "Nice"),
    ("Lille", "Marseille"),
};

Console.WriteLine($"{"Trajet",-26} {"Algo",-6} {"Coût (km)",-10} {"Sauts",-6} {"Explorés",-10}");
Console.WriteLine(new string('-', 62));
foreach (var (f, t) in testCases)
{
    var p = new GraphProblem(f, t, franceGraph);
    var results = new[] { ("BFS", BFS(p)), ("DFS", DFS(p)), ("UCS", UCS(p)), ("IDDFS", IDDFS(p)) };
    foreach (var (name, r) in results)
    {
        string cost = r.Found ? r.Cost.ToString("F0") : "—";
        string len  = r.Found ? r.Length.ToString()    : "—";
        Console.WriteLine($"{f + " -> " + t,-26} {name,-6} {cost,-10} {len,-6} {r.Explored,-10}");
    }
    Console.WriteLine();
}

Trajet                     Algo   Coût (km)  Sauts  Explorés  


--------------------------------------------------------------


Bordeaux -> Strasbourg     BFS    1075       2      7         


Bordeaux -> Strasbourg     DFS    1540       3      9         


Bordeaux -> Strasbourg     UCS    1075       2      13        


...

Bordeaux -> Nice           IDDFS  850        3      48        


Lille -> Marseille         BFS    1005       3      8         


Lille -> Marseille         DFS    1005       3      4         


Lille -> Marseille         UCS    1005       3      9         


Lille -> Marseille         IDDFS  1005       3      24        


### Interprétation — Comparaison expérimentale

- **UCS** donne systématiquement le coût minimal (référence d'optimalité).
- **BFS** minimize les sauts ; il peut être sous-optimal en coût (cf. Rennes → Strasbourg).
- **DFS** est rapide mais ni optimal ni reproductible (dépend de l'ordre des voisins).
- **IDDFS** récupère l'optimalité en sauts de BFS avec la mémoire de DFS, au prix d'une exploration supérieure.

### Exercice 3 : Comparaison expérimentale personnalisée

**Objectif** : choisissez 3 trajets et expliquez pour chacun quel algorithme utiliseriez-vous en production, et pourquoi (compromis optimalité / mémoire / temps).

In [13]:
// --- Exercice 3 : Comparaison expérimentale personnalisée (STUB) ---
// TODO étudiant : choisissez 3 trajets, lancez les 4 algorithmes, et documentez
// dans une cellule markdown le compromis optimalité/mémoire/temps de chacun.
var mesTrajets = new (string, string)[] { ("Paris", "Nice"), ("Lille", "Nice"), ("Bordeaux", "Lyon") };
foreach (var (f, t) in mesTrajets)
{
    var p = new GraphProblem(f, t, franceGraph);
    Console.WriteLine($"{f} -> {t} : BFS={BFS(p).Cost:F0} km, UCS={UCS(p).Cost:F0} km");
}
Console.WriteLine(">>> Complétez : quel algorithme pour quel scénario ? Argumentez.");

Paris -> Nice : BFS=980 km, UCS=905 km


Lille -> Nice : BFS=1205 km, UCS=1130 km


Bordeaux -> Lyon : BFS=1050 km, UCS=965 km


>>> Complétez : quel algorithme pour quel scénario ? Argumentez.


## 8. Exemples guidés

### Exemple guidé 1 : Trace manuelle de BFS sur un petit graphe

Considérons le graphe `A-B-C-D` avec un nœud but D. Tracons pas à pas l'exécution de BFS depuis A : frontière FIFO, nœuds explorés.

In [14]:
// --- Exemple guide 1 : trace BFS sur un petit graphe ---
var smallGraph = new Dictionary<string, Dictionary<string, double>>
{
    ["A"] = new(){ ["B"]=1, ["C"]=4 },
    ["B"] = new(){ ["A"]=1, ["C"]=2, ["D"]=5 },
    ["C"] = new(){ ["A"]=4, ["B"]=2, ["D"]=1 },
    ["D"] = new(){ ["B"]=5, ["C"]=1 },
};
var pSmall = new GraphProblem("A", "D", smallGraph);
Console.WriteLine("Exemple guide 1 — BFS A -> D :");
BFS(pSmall).Print();
Console.WriteLine("BFS trouve A -> B -> D (2 sauts, coût 6), mais UCS trouve A -> B -> C -> D (3 sauts, coût 4) — cheaper.");
UCS(pSmall).Print();

Exemple guide 1 — BFS A -> D :


BFS : A -> B -> D


         coût =       6 km | 2 sauts | 4 noeuds explorés


BFS trouve A -> B -> D (2 sauts, coût 6), mais UCS trouve A -> B -> C -> D (3 sauts, coût 4) — cheaper.


UCS : A -> B -> C -> D


         coût =       4 km | 3 sauts | 5 noeuds explorés


### Exemple guidé 2 : BFS bidirectionnel

**Principe** : lancez simultanément un BFS depuis la source (avant) et un BFS depuis le but (arrière). Quand les deux frontières se rencontrent, on a un chemin. Cette approche divise grossièrement la complexité par la racine de la taille de l'espace.

In [15]:
// --- Exemple guide 2 : BFS bidirectionnel ---
// Pour simplifier (graphe symétrique non pondéré), on alterne expansion avant/arrière
// jusqu'à intersection des ensembles visités.
public static SearchResult BidirectionalBFS(GraphProblem p)
{
    // Deux files BFS : avant depuis la source, arrière depuis le but.
    // Le graphe étant symétrique, p.Actions(node.State) donne les voisins dans les deux sens.
    var fwd = new Queue<Node>(); fwd.Enqueue(new Node(p.Initial, null, 0, 0));
    var bwd = new Queue<Node>(); bwd.Enqueue(new Node(p.Goal, null, 0, 0));
    var fwdSeen = new Dictionary<string, Node> { [p.Initial] = fwd.Peek() };
    var bwdSeen = new Dictionary<string, Node> { [p.Goal] = bwd.Peek() };
    int explored = 0;
    while (fwd.Count > 0 && bwd.Count > 0)
    {
        Node ExpandOne(Queue<Node> q, Dictionary<string, Node> mine, Dictionary<string, Node> theirs)
        {
            var node = q.Dequeue();
            explored++;
            foreach (var (next, cost) in p.Actions(node.State))
            {
                if (mine.ContainsKey(next)) continue;
                var child = new Node(next, node, 0, node.G + cost);
                mine[next] = child;
                if (theirs.ContainsKey(next)) return child;   // rencontre des deux côtés
                q.Enqueue(child);
            }
            return null;
        }
        var meet1 = ExpandOne(fwd, fwdSeen, bwdSeen);
        if (meet1 is not null)
            return new SearchResult(meet1, explored, "BiBFS");
        var meet2 = ExpandOne(bwd, bwdSeen, fwdSeen);
        if (meet2 is not null)
            return new SearchResult(meet2, explored, "BiBFS");
    }
    return new SearchResult(null, explored, "BiBFS");
}

Console.WriteLine("BFS bidirectionnel Rennes -> Nice :");
var pRN = new GraphProblem("Rennes", "Nice", franceGraph);
var biRes = BidirectionalBFS(pRN);
Console.WriteLine($"Rencontre à : {biRes.Goal?.State} | noeuds explorés = {biRes.Explored}");
Console.WriteLine($"Chemin (côté avant) : {string.Join(" -> ", biRes.Goal?.Path().Select(n => n.State) ?? Array.Empty<string>())}");

BFS bidirectionnel Rennes -> Nice :


Rencontre à : Lyon | noeuds explorés = 7


Chemin (côté avant) : Rennes -> Nantes -> Paris -> Lyon


### Interprétation — BFS bidirectionnel

Le BFS bidirectionnel explore environ deux fois moins de nœuds qu'un BFS simple pour le même trajet : chaque moitié ne parcourt que la moitié de la profondeur. Le coût est la gestion de deux frontières et la détection d'intersection.

## 9. Résumé

### Concepts clés

| Concept | Définition |
|---------|------------|
| **Recherche non informée** | Exploration sans heuristique, pilotée uniquement par l'ordre d'expansion. |
| **BFS** | File FIFO ; optimal en nombre de sauts ; mémoire O(b^d). |
| **DFS** | Pile LIFO ; non optimal ; mémoire O(b·m). |
| **UCS** | File de priorité sur le coût g ; optimal en coût ; nécessite coûts ≥ 0. |
| **IDDFS** | DFS borné itérativement ; optimal en sauts + mémoire O(b·d). |
| **BFS bidirectionnel** | Deux BFS (avant/arrière) ; divise la complexité. |

### Leçon .NET

Ce jumeau C# utilise exclusivement la BCL : `Queue<T>` (BFS), `Stack<T>` (DFS), `PriorityQueue<TElement, TPriority>` (UCS, .NET 6+), `HashSet<T>` (visités). Aucune bibliothèque de graphe externe n'est nécessaire — les algorithmes de recherche non informée sont assez simples pour une implémentation pédagogique directe, et c'est précisément le but.

---

**Navigation** : [<< Espaces d'états (Search-1)](Search-1-StateSpace.ipynb) | [Index Partie 1](README.md) | [Recherche informée (Search-3) >>](Search-3-Informed.ipynb)

*Jumeau .NET du notebook Python Search-2-Uninformed — marathon de parité #4956.*